# SQLite | Banco Control #

In [2]:
import sqlite3 as s
import csv

# criação do banco e cursor dos objetos do banco de dados
con = s.connect('controls.db')
cur = con.cursor()

In [3]:
# Criação da tabela despesas
cur.execute("""
    CREATE TABLE despesas(
        id INTEGER PRIMARY KEY,
        data DATE,
        categoria VARCHAR(100),
        descricao VARCHAR(100), 
        valor REAL,
        forma_pagamento VARCHAR(50)
    )
""")

In [4]:
# 1. Abra o arquivo CSV usando a biblioteca csv
path = '/home/rlbarbosa/Projects/sqlite3_analysis/raw/despesas.csv'

with open(path, mode='r', encoding='utf-8') as arquivo:
    leitor = csv.reader(arquivo)
    
    # 2. Pule o cabeçalho se o seu arquivo tiver os nomes das colunas na 1ª linha
    next(leitor) 
    
    # 3. O leitor converte cada linha em uma lista com os valores separados: ["col1", "col2", ...]
    cur.executemany("""
        INSERT INTO despesas
        VALUES (?, ?, ?, ?, ?, ?)
    """, leitor)

con.commit()

# Análises das Despesas #

### Abaixo segue sete questionamentos focados em gestão financeira e fluxo de caixa, estruturados para demonstrar domínio analítico e técnico na construção do seu portfólio no GitHub ###

1. Análise de Centro de Custos (Categorias): Qual é o valor total consolidado por categoria e qual delas representa a maior fatia das despesas no período analisado?   

2. Conciliação de Meios de Pagamento: Qual é a distribuição (em valor absoluto e percentual) das despesas por forma_pagamento, como Pix, Dinheiro, Cartões e Boleto?

3. Fluxo de Caixa Diário: Qual o volume total de saídas financeiras agrupado por data ao longo da primeira semana de janeiro de 2023? 

4. Ticket Médio por Transação: Qual é o valor médio geral das transações e como esse ticket médio varia quando analisado isoladamente por cada forma de pagamento listada no arquivo print_retorno_query.png? 

5. Curva ABC de Gastos: Quais foram as 3 maiores despesas individuais (ordenadas pelo maior valor) registradas na base, e a quais descrições e categorias elas pertencem? 

6. Detalhamento de Centro de Custo Específico: Considerando apenas a categoria "Lazer", qual o montante total gasto e qual a representatividade percentual de cada item da descricao (Jogos, Streaming, Bar) dentro dessa categoria? 

7. Liquidez e Rastreabilidade: Qual o montante total e a quantidade de transações pagas exclusivamente em "Dinheiro" (que possuem menor rastreabilidade bancária) em comparação ao somatório dos meios digitais?

### Exploração de dados ###

In [5]:
# Exploração dos dados

import pandas as pd

df = pd.read_sql_query("SELECT * FROM despesas", con)
df.head(10)

,id,data,categoria,descricao,valor,forma_pagamento
0,1,2023-01-02,Lazer,Jogos,129.76,Pix
1,2,2023-01-02,Alimentação,Supermercado,110.30,Cartão de débito
2,3,2023-01-02,Transporte,Aplicativo de transporte,108.26,Dinheiro
3,4,2023-01-02,Serviços,Lavanderia,31.52,Dinheiro
4,5,2023-01-03,Lazer,Streaming,188.50,Pix
5,6,2023-01-04,Educação,Plataforma online,270.33,Dinheiro
6,7,2023-01-05,Transporte,Pedágio,142.49,Dinheiro
7,8,2023-01-05,Serviços,Salão de beleza,218.99,Cartão de crédito
8,9,2023-01-06,Outros,Doação,244.70,Dinheiro
9,10,2023-01-06,Lazer,Bar,255.77,Boleto


In [6]:
# Colunas e registros 
print(f'Na fonte de dados temos {df.shape[1]} colunas e {df.shape[0]} registros')

Na fonte de dados temos 6 colunas e 2200 registros


In [7]:
# Tipagem dos dados 
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               2200 non-null   int64  
 1   data             2200 non-null   str    
 2   categoria        2200 non-null   str    
 3   descricao        2200 non-null   str    
 4   valor            2200 non-null   float64
 5   forma_pagamento  2200 non-null   str    
dtypes: float64(1), int64(1), str(4)
memory usage: 103.3 KB


In [ ]:
# Identificar se há series nulas
df.isnull().any().any()

np.False_

### 1. Análise de centro de cursos ###

In [10]:
df

,id,data,categoria,descricao,valor,forma_pagamento
0,1,2023-01-02,Lazer,Jogos,129.76,Pix
1,2,2023-01-02,Alimentação,Supermercado,110.30,Cartão de débito
2,3,2023-01-02,Transporte,Aplicativo de transporte,108.26,Dinheiro
3,4,2023-01-02,Serviços,Lavanderia,31.52,Dinheiro
4,5,2023-01-03,Lazer,Streaming,188.50,Pix
...,...,...,...,...,...,...
2195,2196,2024-12-27,Alimentação,Supermercado,53.86,Cartão de débito
2196,2197,2024-12-29,Lazer,Streaming,228.57,Pix
2197,2198,2024-12-29,Moradia,Internet,888.92,Cartão de débito
2198,2199,2024-12-31,Transporte,Estacionamento,100.29,Pix


In [11]:
df_total = len(df)

df1 = pd.DataFrame(df['categoria'].value_counts())
df1 = df1.rename(columns={"count": "qtde"})
df1['percent'] = round(df1['qtde']/df_total, 2)

df1

,qtde,percent
categoria,,
Alimentação,486,0.22
Transporte,362,0.16
Moradia,277,0.13
Lazer,227,0.10
Saúde,210,0.10
Educação,190,0.09
Vestuário,164,0.07
Serviços,151,0.07
Outros,133,0.06
